# Origem dos dados

Dados de clima:
- Stations: [NYC]
- Available Data: Air Temperature, Relative Humidity, Wind Direction, Wind Speed [mph], 1 hour Preciptation [mm]
- Timezone of Observation Times: America/New_York (EST/EDT)
- Dataset de treino do imputer: 2017, 2018, 2019, 2020, 2021
- Dataset de treino: 2022
- Dataset de teste: 2023
- Fonte: https://mesonet.agron.iastate.edu/request/download.phtml?network=NY_ASOS

Dados taxi 2022:
- Fonte: https://data.cityofnewyork.us/Transportation/2022-Green-Taxi-Trip-Data/8nfn-ifaj/about_data

Dados taxi 2023:
- Fonte: https://data.cityofnewyork.us/Transportation/2023-Green-Taxi-Trip-Data/peyi-gg4n/about_data

Zonas:
- Fonte: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page


- Venue Escolhida: BreSci

Para o artigo:
- Resultado com grid search
- Resultado com baesyan search
- Resultado com teste empirico
- O modelo utilizado é XGBRegressor.
- Foram testadas os hiperparametros:
    - n_estimators
    - learning_rate
    - max_depth
    - subsample
    - min_child_weight
    - colsample_bynode
    - colsample_bytree
    - colsample_bylevel
    - num_parallel_tree
    - tree_method
    - max_delta_step
    - random_state
    - gamma


- Os seguintes não foram eficazes, por piorar o modelo ou por causar pouco efeito
    - colsample_bynode
    - colsample_bytree
    - colsample_bylevel
    - tree_method
    - max_delta_step
    - gamma



- O hiperparametro random_state não causou efeito mas foi mantido para manter a reprodutibilidade

ACMDL

# Questões

- Tratamento do clima OK
- Verificar quais os dados necessários tratar nos dataset de taxi OK
- Realizar o tratamento do dataset do taxi OK
- Mensurar quantas linhas foram retiradas do dataset
- Juntar os dois datasets OK
- Criar as questões a serem respondidas:
    - Prever a quatidade de corrida se baseando no clima e em feriados e fins de semana


In [1]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import holidays

In [2]:
# Carregando datasets

teste = 0 #limitador para facilitar o desenvolvimento
if teste == 0:
    treinamentoClima = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/ClimaTreino.csv')
    Clima = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/Clima2023.csv')
    Clima2022 = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/Clima2022.csv')
    taxiTreino = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/2022_Green_Taxi_Trip_Data_20260530.csv')
    taxi_2023 = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/2023_Green_Taxi_Trip_Data_20260519.csv')
else:
    treinamentoClima = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/ClimaTreino.csv', nrows=2000)
    Clima = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/Clima2023.csv', nrows=5000)
    Clima2022 = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/Clima2022.csv', nrows=5000)
    taxiTreino = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/2022_Green_Taxi_Trip_Data_20260530.csv', nrows=5000)
    taxi_2023 = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/2023_Green_Taxi_Trip_Data_20260519.csv', nrows=5000)


C:\Users\jhrat\AppData\Local\Temp\ipykernel_6332\1262410919.py:8: DtypeWarning: Columns (8,9,16) have mixed types. Specify dtype option on import or set low_memory=False.
  taxiTreino = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/2022_Green_Taxi_Trip_Data_20260530.csv')
C:\Users\jhrat\AppData\Local\Temp\ipykernel_6332\1262410919.py:9: DtypeWarning: Columns (3,8,9,16) have mixed types. Specify dtype option on import or set low_memory=False.
  taxi_2023 = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/2023_Green_Taxi_Trip_Data_20260519.csv')


In [3]:
quantidade_linhas_taxiTreino = len(taxiTreino)
quantidade_linhas_taxi = len(taxi_2023)
quantidade_linhas_climaTreino = len(Clima2022)
quantidade_linhas_clima = len(Clima)

In [4]:
# Tratamento de dataset clima

# Alterando nome das colunas
treinamentoClima.rename(columns={'valid': 'data', 'tmpc': 'temperatura [C°]', 'relh': 'umidade relativa [%]', 'drct': 'direção do vento [°]', 'sped': 'velocidade do vento [mph]', 'p01m': 'precipitação acumulada em 1 hora [mm]', }, inplace=True)
Clima2022.rename(columns={'valid': 'data', 'tmpc': 'temperatura [C°]', 'relh': 'umidade relativa [%]', 'drct': 'direção do vento [°]', 'sped': 'velocidade do vento [mph]', 'p01m': 'precipitação acumulada em 1 hora [mm]', }, inplace=True)
Clima.rename(columns={'valid': 'data', 'tmpc': 'temperatura [C°]', 'relh': 'umidade relativa [%]', 'drct': 'direção do vento [°]', 'sped': 'velocidade do vento [mph]', 'p01m': 'precipitação acumulada em 1 hora [mm]', }, inplace=True)

# ajuste de datas
treinamentoClima['data'] = pd.to_datetime(treinamentoClima['data'])
Clima2022['data'] = pd.to_datetime(Clima2022['data'])
Clima['data'] = pd.to_datetime(Clima['data'])

# Trocando valores vazios para nulos
valores_para_nulo = {
    'M': np.nan,
    'T': np.nan,
    ' ': np.nan,
    '': np.nan,
    'N/A': np.nan
}

# Aplica a substituição no dataset
treinamentoClima.replace(valores_para_nulo, inplace=True)
Clima2022.replace(valores_para_nulo, inplace=True)
Clima.replace(valores_para_nulo, inplace=True)

# Ordena o Dataset
Clima2022 = Clima2022.sort_values(by='data', ascending=True)
Clima = Clima.sort_values(by='data', ascending=True)

# Refaz a contagem das linhas
Clima2022 = Clima2022.reset_index(drop=True)
Clima = Clima.reset_index(drop=True)

In [5]:
# Tratamento dataset taxi

# Apagar colunas sem uso
colunas_descate = ['store_and_fwd_flag', 'ehail_fee', 'fare_amount', 'extra', 'trip_type', 'VendorID']
taxi_2023 = taxi_2023.drop(columns=colunas_descate)
taxiTreino = taxiTreino.drop(columns=colunas_descate)

# Apaga corridas com trip_distance = zero ou NaN
taxi_2023 = taxi_2023.dropna(subset=['trip_distance'])
taxi_2023 = taxi_2023[taxi_2023['trip_distance'] != 0]
taxiTreino = taxiTreino.dropna(subset=['trip_distance'])
taxiTreino = taxiTreino[taxiTreino['trip_distance'] != 0]

# Apaga corridas com zero ou NaN passageiros
taxi_2023 = taxi_2023.dropna(subset=['passenger_count'])
taxi_2023 = taxi_2023[taxi_2023['passenger_count'] != 0]
taxiTreino = taxiTreino.dropna(subset=['passenger_count'])
taxiTreino = taxiTreino[taxiTreino['passenger_count'] != 0]

# Converte a string para data
taxi_2023['lpep_pickup_datetime'] = pd.to_datetime(taxi_2023['lpep_pickup_datetime'])
taxi_2023['lpep_dropoff_datetime'] = pd.to_datetime(taxi_2023['lpep_dropoff_datetime'])
taxiTreino['lpep_pickup_datetime'] = pd.to_datetime(taxiTreino['lpep_pickup_datetime'])
taxiTreino['lpep_dropoff_datetime'] = pd.to_datetime(taxiTreino['lpep_dropoff_datetime'])

# Converte valores string em float da coluna trip_distance
taxi_2023['trip_distance'] = taxiTreino['trip_distance'].astype(str).str.replace(',', '')
taxi_2023['trip_distance'] = pd.to_numeric(taxiTreino['trip_distance'], errors='coerce')
taxiTreino['trip_distance'] = taxiTreino['trip_distance'].astype(str).str.replace(',', '')
taxiTreino['trip_distance'] = pd.to_numeric(taxiTreino['trip_distance'], errors='coerce')

# Converte valores string em float da coluna total_amount
taxi_2023['total_amount'] = taxiTreino['total_amount'].astype(str).str.replace(',', '')
taxi_2023['total_amount'] = pd.to_numeric(taxiTreino['total_amount'], errors='coerce')
taxiTreino['total_amount'] = taxiTreino['total_amount'].astype(str).str.replace(',', '')
taxiTreino['total_amount'] = pd.to_numeric(taxiTreino['total_amount'], errors='coerce')

# Apaga a linha quando total_amount for NaN
taxi_2023 = taxi_2023.dropna(subset=['total_amount'])
taxiTreino = taxiTreino.dropna(subset=['total_amount'])

# Apaga a linha quando houver valores maiores que 15000 na coluna trip_distance
taxi_2023 = taxi_2023.dropna(subset=['trip_distance'])
taxi_2023 = taxi_2023[taxi_2023['trip_distance'] < 700]
taxiTreino = taxiTreino.dropna(subset=['trip_distance'])
taxiTreino = taxiTreino[taxiTreino['trip_distance'] < 700]

# Mantém no dataset somente corridas do ano indicado no dataset
taxi_2023 = taxi_2023[taxi_2023['lpep_pickup_datetime'].dt.year == 2023]
taxiTreino = taxiTreino[taxiTreino['lpep_pickup_datetime'].dt.year == 2022]

# Ordena as linhas em ordem crescente
taxi_2023 = taxi_2023.sort_values(by='lpep_pickup_datetime', ascending=True)
taxiTreino = taxiTreino.sort_values(by='lpep_pickup_datetime', ascending=True)

# Refaz a contagem das linhas
taxi_2023 = taxi_2023.reset_index(drop=True)
taxiTreino = taxiTreino.reset_index(drop=True)


In [6]:
# Treinamento do Imputer

# Colunas a serem analisadas
colunas_alvo = ['temperatura [C°]', 'umidade relativa [%]', 'direção do vento [°]', 'velocidade do vento [mph]', 'precipitação acumulada em 1 hora [mm]']

climaCopia = Clima.copy()
clima2022Copia = Clima2022.copy()

imputerTeste = 0
if imputerTeste == 0:
    # Caracteristicas do imputer
    imputer = IterativeImputer(random_state=85,
                                max_iter=80,
                                initial_strategy='mean')
elif imputerTeste == 1:
    # Caracteristicas do imputer
    imputer = IterativeImputer(random_state=85,
                                max_iter=80,
                                initial_strategy='most_frequent',
                                estimator=RandomForestRegressor(n_estimators=20, random_state=42, n_jobs=-1))
elif imputerTeste == 2:
    import xgboost as xgb
    XGBoost = xgb.XGBRegressor(n_estimators=50, tree_method='hist', n_jobs=-1, random_state=42)
    imputer = IterativeImputer(
                    estimator=XGBoost, 
                    max_iter=600,
                    tol=1e-2,
                    random_state=42)
elif imputerTeste == 3:
    from sklearn.impute import KNNImputer
    imputer = KNNImputer(n_neighbors=5, weights='distance')
    
    
# Realizando o treinamento com o dataset de treinamento
imputer.fit(treinamentoClima[colunas_alvo])

# Executa o algoritmo treinado e recebe as colunas preenchidas
valores_clima = imputer.transform(climaCopia[colunas_alvo])
valores_clima2022 = imputer.transform(clima2022Copia[colunas_alvo])

# Coloca as colunas preenchidas no dataset
climaCopia[colunas_alvo] = valores_clima
clima2022Copia[colunas_alvo] = valores_clima2022

Clima.update(climaCopia)
Clima2022.update(clima2022Copia)

In [7]:
# Análise dos datasets
analisar = 0



meus_datasets = {
    'Treinamento Clima (2020-2022)': treinamentoClima,
    'Clima Teste (2023)': Clima,
    'Táxi Treino (2022)': taxiTreino,
    'Táxi Teste (2023)': taxi_2023
}

if analisar == 1:
    for nome_do_dataset, df in meus_datasets.items():
        print(f"\n{'='*65}")
        print(f" A analisar o dataset: {nome_do_dataset}")
        print(f"{'='*65}")
    

        for coluna in df.columns:
            tipo_dado = df[coluna].dtype
            qtd_nulos = df[coluna].isna().sum()
            print(f"Coluna: {coluna:<35} | Tipo: {str(tipo_dado):<15} | Nulos: {qtd_nulos}")



    for nome_do_dataset, df in meus_datasets.items():
        print(f"\n{'='*80}")
        print(f"Analisando os tipos REAIS do dataset: {nome_do_dataset}")
        print(f"{'='*80}")
    

        for coluna in df.columns:
        

            contagem_tipos = df[coluna].apply(lambda x: type(x).__name__).value_counts()
        
            # Formata os tipos e quantidades numa linha de texto legível
            tipos_str = ", ".join([f"{tipo}: {qtd} linhas" for tipo, qtd in contagem_tipos.items()])
        
            # Se a coluna tiver mais de 1 tipo (dados misturados), cria um alerta!
            alerta = "MISTO!" if len(contagem_tipos) > 1 else ""
        
            # Imprime o relatório da coluna
            print(f"Coluna: {coluna:<35} | Contém -> {tipos_str}{alerta}")

In [8]:
# Unificar datasets taxi2022 e clima2022

# Cria no taxi as colunas de interesse do clima 
taxiTreino['temperatura [C°]'] = np.nan
taxiTreino['umidade relativa [%]'] = np.nan
taxiTreino['direção do vento [°]'] = np.nan
taxiTreino['velocidade do vento [mph]'] = np.nan
taxiTreino['precipitação acumulada em 1 hora [mm]'] = np.nan

# Unifica o dataset taxi e o clima
indexClima = 0
indextaxi = 0
index_taxi = indextaxi
for index_taxi, linha_taxi in taxiTreino.iterrows():
    data_corrida = linha_taxi['lpep_pickup_datetime']
    
    # Procura o index ideal para preencher os espaços vazios
    indexLocal_clima = indexClima
    for indexLocal_clima, linha_clima in Clima2022.iloc[indexClima:].iterrows():
        data_clima = linha_clima['data']
        if data_clima > data_corrida:
            if indexLocal_clima > 0:
                indexClima = indexLocal_clima - 1
            else:
                indexClima = indexLocal_clima
            break
    
    # Preenche os espaços vazios
    taxiTreino.loc[index_taxi, 'temperatura [C°]'] = Clima2022['temperatura [C°]'].iloc[indexClima]
    taxiTreino.loc[index_taxi, 'umidade relativa [%]'] = Clima2022['umidade relativa [%]'].iloc[indexClima]
    taxiTreino.loc[index_taxi, 'direção do vento [°]'] = Clima2022['direção do vento [°]'].iloc[indexClima]
    taxiTreino.loc[index_taxi, 'velocidade do vento [mph]'] = Clima2022['velocidade do vento [mph]'].iloc[indexClima]
    taxiTreino.loc[index_taxi, 'precipitação acumulada em 1 hora [mm]'] = Clima2022['precipitação acumulada em 1 hora [mm]'].iloc[indexClima]

In [9]:
# Unificar datasets taxi2023 e clima2023
# Garantindo que os datasets estão ordenados pela data
taxi_2023 = taxi_2023.sort_values(by='lpep_pickup_datetime', ascending=True)
Clima = Clima.sort_values(by='data', ascending=True)

# Cria no taxi as colunas de interesse do clima 
taxi_2023['temperatura [C°]'] = np.nan
taxi_2023['umidade relativa [%]'] = np.nan
taxi_2023['direção do vento [°]'] = np.nan
taxi_2023['velocidade do vento [mph]'] = np.nan
taxi_2023['precipitação acumulada em 1 hora [mm]'] = np.nan

# Unifica o dataset taxi e o clima
indexClima = 0
index_taxi = 0
for index_taxi, linha_taxi in taxi_2023.iterrows():
    data_corrida = linha_taxi['lpep_pickup_datetime']
    
    # Procura o index ideal para preencher os espaços vazios
    indexLocal_clima = indexClima
    for indexLocal_clima, linha_clima in Clima.iloc[indexClima:].iterrows():
        data_clima = linha_clima['data']
        if data_clima > data_corrida:
            if indexLocal_clima > 0:
                indexClima = indexLocal_clima - 1
            else:
                indexClima = indexLocal_clima
            break

    
    
    # Preenche os espaços vazios
    taxi_2023.loc[index_taxi, 'temperatura [C°]'] = Clima['temperatura [C°]'].iloc[indexClima]
    taxi_2023.loc[index_taxi, 'umidade relativa [%]'] = Clima['umidade relativa [%]'].iloc[indexClima]
    taxi_2023.loc[index_taxi, 'direção do vento [°]'] = Clima['direção do vento [°]'].iloc[indexClima]
    taxi_2023.loc[index_taxi, 'velocidade do vento [mph]'] = Clima['velocidade do vento [mph]'].iloc[indexClima]
    taxi_2023.loc[index_taxi, 'precipitação acumulada em 1 hora [mm]'] = Clima['precipitação acumulada em 1 hora [mm]'].iloc[indexClima]

In [11]:
def gerar_lista_datas_especiais(ano):
    # Instancia os feriados oficiais do Estado de NY para o ano especificado
    feriados_ny = holidays.US(subdiv='NY', years=ano)
    
    datas_especiais = set()
    
    # Define o primeiro e o último dia do ano
    data_inicial = datetime.date(ano, 1, 1)
    data_final = datetime.date(ano, 12, 31)
    
    delta = datetime.timedelta(days=1)
    data_atual = data_inicial
    
    # Percorre todos os dias do ano
    while data_atual <= data_final:
        # Verifica se é Sábado (5), Domingo (6) OU se está na lista de feriados
        if data_atual.weekday() >= 5 or data_atual in feriados_ny:
            datas_especiais.add(data_atual)
        data_atual += delta
        
    # Ordena as datas cronologicamente
    datas_ordenadas = sorted(list(datas_especiais))
    
    return datas_ordenadas

In [12]:
datas_especiais_2022 = gerar_lista_datas_especiais(2022)
datas_especiais_2023 = gerar_lista_datas_especiais(2023)

In [13]:
# Dataset depois da união
quantidade_linhas_taxiTreino_clima = len(taxiTreino)
quantidade_linhas_taxi_clima = len(taxi_2023)

taxiTreino_final = quantidade_linhas_taxiTreino - quantidade_linhas_taxiTreino_clima
taxi_final = quantidade_linhas_taxi - quantidade_linhas_taxi_clima

print(f"O dataset de CLIMA TREINO tem {quantidade_linhas_taxiTreino_clima} linhas.")
print(f"O dataset de CLIMA TESTE tem {quantidade_linhas_taxiTreino} linhas.")
print(f"===========================================================")
print(f"===========================================================")
print(f"O dataset de TAXI TREINO tem {quantidade_linhas_taxiTreino} linhas.")
print(f"Após o agrupamento: {quantidade_linhas_taxiTreino_clima} linhas.")
print(f"Foram retidaras {taxiTreino_final} linhas.")
print(f"===========================================================")
print(f"===========================================================")
print(f"O dataset de TAXI TESTE tem {quantidade_linhas_taxi} linhas.")
print(f"Após o agrupamento: {quantidade_linhas_taxi_clima} linhas.")
print(f"Foram retidaras {taxi_final} linhas.")

O dataset de CLIMA TREINO tem 734172 linhas.
O dataset de CLIMA TESTE tem 840402 linhas.
O dataset de TAXI TREINO tem 840402 linhas.
Após o agrupamento: 734172 linhas.
Foram retidaras 106230 linhas.
O dataset de TAXI TESTE tem 787060 linhas.
Após o agrupamento: 640265 linhas.
Foram retidaras 146795 linhas.


In [14]:
# Adicionar finais de semana e feriados no dataset de taxi

# Cria a coluna de feriados/fins de semana com valores nulos
taxi_2023['feriado/fim de semana'] = np.nan
taxiTreino['feriado/fim de semana'] = np.nan

# Extrai das datas
datas_corridas_2023 = taxi_2023['lpep_pickup_datetime'].dt.date
datas_corridas_treino = taxiTreino['lpep_pickup_datetime'].dt.date

# Coloca 1 em feriados e final de semana e zero para outros dias
taxi_2023['feriado/fim de semana'] = np.where(datas_corridas_2023.isin(datas_especiais_2023), 1, 0)
taxiTreino['feriado/fim de semana'] = np.where(datas_corridas_treino.isin(datas_especiais_2022), 1, 0)

# Salva as alteraçãoes
taxi_2023.to_csv('taxi_clima.csv', index=False)
taxiTreino.to_csv('taxiTreino_clima.csv', index=False)